# Retail Sales – Exploratory Data Analysis (EDA)

**Objective:** Explore retail transactions to understand sales trends, customer demographics, product performance, and relationships between numerical variables.

**Dataset:** `retail_sales.csv` (included sample dataset; replace it with your own file using the same column names).

> The included dataset is synthetic and is intended for learning/demo purposes. Findings from it are not real-world business facts.


## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)


## 2. Load the dataset and inspect its structure

In [ ]:
df = pd.read_csv("retail_sales.csv")

print("Dataset shape:", df.shape)
display(df.head())
df.info()


## 3. Check missing values and duplicates

In [ ]:
print("Missing values by column:")
display(df.isna().sum().to_frame("missing_count"))

print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().copy()

# Convert data types safely
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
for col in ["Age", "Quantity", "Price per Unit", "Total Amount"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Rows after removing duplicates:", len(df))


**Observation:** Review the missing-value counts above. Duplicate rows are removed, and the date and numeric fields are converted to appropriate data types. If your real dataset contains missing values, decide whether to fill or remove them based on the business context.

## 4. Descriptive statistics

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns

summary = pd.DataFrame({
    "mean": df[numeric_cols].mean(),
    "median": df[numeric_cols].median(),
    "mode": df[numeric_cols].mode().iloc[0],
    "std_dev": df[numeric_cols].std(),
    "min": df[numeric_cols].min(),
    "max": df[numeric_cols].max()
})
display(summary)


## 5. Prepare time features

In [ ]:
df = df.dropna(subset=["Date"]).copy()
df["Month"] = df["Date"].dt.to_period("M")
df["Quarter"] = df["Date"].dt.to_period("Q")


## 6. Monthly sales trend

In [ ]:
monthly_sales = df.groupby("Month")["Total Amount"].sum()
ax = monthly_sales.plot(marker="o")
ax.set(title="Monthly Retail Sales", xlabel="Month", ylabel="Revenue")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


**Observation:** Identify months with the highest and lowest revenue and check whether there is a recurring seasonal pattern. Plan stock and campaigns around observed peaks; do not assume a seasonal pattern from a short dataset.

## 7. Quarterly sales trend

In [ ]:
quarterly_sales = df.groupby("Quarter")["Total Amount"].sum()
ax = quarterly_sales.plot(marker="o")
ax.set(title="Quarterly Retail Sales", xlabel="Quarter", ylabel="Revenue")
plt.tight_layout()
plt.show()


**Observation:** Compare quarter-to-quarter revenue. A sustained increase may indicate growth, while a decline should prompt investigation of pricing, availability, and customer demand.

## 8. Customer age-group distribution

In [ ]:
age_bins = [0, 18, 25, 35, 45, 60, 100]
age_labels = ["Under 18", "18–25", "26–35", "36–45", "46–60", "60+"]
df["Age Group"] = pd.cut(df["Age"], bins=age_bins, labels=age_labels, right=True)

age_order = age_labels
sns.countplot(data=df, x="Age Group", order=age_order)
plt.title("Customer Age-Group Distribution")
plt.xlabel("Age group")
plt.ylabel("Transactions")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()


**Observation:** The largest bar represents the age group with the most transactions. Use this as a starting point for customer segmentation, while remembering that transaction counts are not the same as unique-customer counts.

## 9. Customer gender breakdown

In [ ]:
gender_counts = df["Gender"].value_counts()
plt.figure(figsize=(7, 7))
plt.pie(gender_counts, labels=gender_counts.index, autopct="%1.1f%%", startangle=90)
plt.title("Transactions by Gender")
plt.ylabel("")
plt.show()


**Observation:** Compare the transaction share across gender groups. Avoid treating transaction share as population share or assuming it explains purchasing preferences by itself.

## 10. Top 10 best-selling products

In [ ]:
top_products = (df.groupby("Product")["Quantity"]
                 .sum().sort_values(ascending=False).head(10))
top_products.sort_values().plot(kind="barh")
plt.title("Top 10 Best-Selling Products by Units")
plt.xlabel("Units sold")
plt.ylabel("Product")
plt.tight_layout()
plt.show()


**Observation:** These products lead by units sold, not necessarily by revenue or profit. Ensure high-demand products are available and review their margins before prioritising them.

## 11. Revenue by product category

In [ ]:
category_revenue = (df.groupby("Product Category")["Total Amount"]
                    .sum().sort_values(ascending=False))
category_revenue.plot(kind="bar", color="steelblue")
plt.title("Revenue by Product Category")
plt.xlabel("Product category")
plt.ylabel("Revenue")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


**Observation:** The tallest bar indicates the category contributing the most revenue. Compare this with units sold to distinguish high-volume categories from higher-priced categories.

## 12. Correlation heatmap

In [ ]:
numeric_df = df.select_dtypes(include=np.number)
corr = numeric_df.corr()
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", center=0)
plt.title("Correlation Between Numerical Variables")
plt.tight_layout()
plt.show()


**Observation:** Correlation describes linear association, not causation. For example, a strong relationship between quantity and total amount is expected because total amount is calculated using quantity and unit price.

## 13. Additional visualisation: average order value by age group

In [ ]:
aov_by_age = df.groupby("Age Group", observed=False)["Total Amount"].mean().reindex(age_labels)
aov_by_age.plot(kind="bar", color="teal")
plt.title("Average Transaction Value by Customer Age Group")
plt.xlabel("Age group")
plt.ylabel("Average transaction value")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()


**Observation:** This chart compares average transaction value across age groups, rather than transaction volume. A group with fewer transactions may still have a higher average transaction value.

## 14. Key metrics

In [ ]:
total_revenue = df["Total Amount"].sum()
total_units = df["Quantity"].sum()
transactions = df["Transaction ID"].nunique()
avg_transaction_value = df.groupby("Transaction ID")["Total Amount"].sum().mean()

print(f"Total revenue: {total_revenue:,.2f}")
print(f"Units sold: {total_units:,}")
print(f"Transactions: {transactions:,}")
print(f"Average transaction value: {avg_transaction_value:,.2f}")


## 15. Conclusion and actionable recommendations

Use the actual charts and metrics above to validate these recommendations before presenting them:

1. **Inventory planning:** Keep adequate stock of the top-selling products and review stock-outs during high-sales months.
2. **Category strategy:** Allocate marketing budget to high-revenue categories, while checking profit margins before increasing promotions.
3. **Customer segmentation:** Tailor campaigns to age groups with higher average transaction values, and test results with controlled campaigns.
4. **Seasonal planning:** Use monthly and quarterly patterns to schedule promotions and replenishment.
5. **Basket-value growth:** Test bundles or complementary-product offers and measure their impact on average transaction value.

**Important:** These are hypotheses to investigate, not conclusions guaranteed by the synthetic sample data. Replace the sample CSV with a real dataset for business decisions.
